# 🍅 Taller de Visión Artificial: Detector de Calidad de Frutas

**Objetivo:** Crear un sistema inteligente capaz de identificar jitomates, evaluar su forma y determinar su madurez mediante una cámara web en tiempo real.

## 1. Importación de Librerías

Para empezar, necesitamos importar las herramientas fundamentales:
* **OpenCV (`cv2`):** Es el "ojo" del sistema. Nos permite capturar video y procesar imágenes.
* **NumPy (`np`):** Es el "cerebro matemático". Las imágenes son matrices de números, y NumPy nos permite calcular distancias de color y formas eficientemente.

In [ ]:
import cv2
import numpy as np
import time
import os

print("Librerías importadas correctamente. Versión OpenCV:", cv2.__version__)

## 2. Configuración y Calibración (El Cerebro del Sistema)

En visión artificial, evitamos usar "números mágicos" dispersos por el código. Agrupamos todos los parámetros en un diccionario de configuración (`CFG`).

### Conceptos Clave:
1.  **Espacio de Color HSV:** A diferencia del RGB (Rojo, Verde, Azul), el HSV separa el color (**Hue**) de la intensidad de luz (**Value**). Esto nos permite detectar un jitomate rojo aunque esté en la sombra.
2.  **Solidez (Solidity):** Una medida de qué tan "compacto" es un objeto. Un jitomate es sólido (cerca de 1.0). El jengibre o una mano abierta tienen huecos, por lo que su solidez es baja.
3.  **Cáliz (Calyx):** La parte verde superior. Es nuestra "huella digital" biológica para distinguir un jitomate rojo de una manzana roja.

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

CFG = {
    # ---------- Visualización ----------
    "TARGET_VIEW_W": 1280,        # Ancho de la ventana
    "FONT": cv2.FONT_HERSHEY_SIMPLEX,

    # ---------- Segmentación (Separar objeto del fondo) ----------
    "KERNEL": 5,           # Tamaño del "pincel" para limpiar ruido
    "OPEN_IT": 2,          # Iteraciones para eliminar puntos blancos pequeños (ruido)
    "CLOSE_IT": 3,         # Iteraciones para cerrar huecos negros dentro de la fruta
    "MIN_AREA": 2500,      # Tamaño mínimo en píxeles para ser considerado fruta
    "MIN_EXTENT": 0.35,    # Relación Área/Rectángulo contenedor
    "BORDER_FRAC": 0.06,   # Margen para detectar el color del fondo automáticamente

    # ---------- Forma: score (Geometría) ----------
    "TOMATO_MIN_SOLIDITY": 0.88, # ¿Qué tan "relleno" es el contorno?
    "TOMATO_MIN_CIRC": 0.42,     # ¿Qué tan círculo es? (1.0 es círculo perfecto)
    "TOMATO_MAX_ASPECT": 1.65,   # Proporción Ancho/Alto (no queremos jitomates muy alargados)
    
    # ---------- El Cáliz (La "hojita" verde superior) ----------
    "USE_CALYX_CHECK": True,
    "CALYX_TOP_FRAC": 0.40,      # Solo buscamos verde en el 40% superior de la fruta
    "CALYX_H_MIN": 35, "CALYX_H_MAX": 95, # Rango de color verde en HSV
    "CALYX_MIN_S": 50, "CALYX_MIN_V": 40, # Mínima saturación y brillo

    # ---------- Color (Rangos HSV para madurez) ----------
    "H_RED1_MAX": 10,     # Rojo parte 1 (0-10)
    "H_RED2_MIN": 165,    # Rojo parte 2 (165-180) - El rojo da la vuelta en el círculo
    "H_ORANGE_MIN": 10, "H_ORANGE_MAX": 25,
    "H_YELLOW_MIN": 25, "H_YELLOW_MAX": 40,
    "H_GREEN_MIN": 35,  "H_GREEN_MAX": 95,
    
    # ---------- Colores Visuales (BGR para dibujar) ----------
    "C_RED": (0, 0, 255),    "C_ORANGE": (0, 165, 255),
    "C_GREEN": (0, 255, 0),  "C_YELLOW": (0, 255, 255),
    "C_GRAY": (30, 30, 30),
}

## 3. Utilidades y Matemáticas de la Forma

Aquí definimos las funciones que calculan la geometría del objeto.

* **Convex Hull:** Imagina poner una liga elástica alrededor del objeto. El área dentro de esa liga comparada con el área real del objeto nos da la "Solidez".
* **Bounding Rect:** El rectángulo más pequeño que encierra al objeto.

In [ ]:
def resize_to_width(img, target_w):
    """Redimensiona la imagen manteniendo la proporción."""
    h, w = img.shape[:2]
    scale = target_w / float(w)
    new_w, new_h = int(w * scale), int(h * scale)
    return cv2.resize(img, (new_w, new_h))

def contour_metrics(cnt):
    """Calcula propiedades geométricas avanzadas de un contorno."""
    area = cv2.contourArea(cnt)
    if area <= 0: return None

    peri = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    
    # Matemáticas de forma
    circ = (4.0 * np.pi * area) / (peri * peri) if peri > 0 else 0
    
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.0
    
    aspect = max(w, h) / max(1, min(w, h))
    extent = area / (w * h)

    return {"area": area, "x": x, "y": y, "w": w, "h": h,
            "circ": circ, "solidity": solidity, "extent": extent, "aspect": aspect}

def tomato_shape_score(metrics):
    """Retorna 0.0 a 1.0 indicando qué tanto parece un jitomate por su forma."""
    s_circ = np.clip((metrics["circ"] - CFG["TOMATO_MIN_CIRC"]) / (1.0 - CFG["TOMATO_MIN_CIRC"]), 0, 1)
    s_sol  = np.clip((metrics["solidity"]  - CFG["TOMATO_MIN_SOLIDITY"]) / (1.0 - CFG["TOMATO_MIN_SOLIDITY"]), 0, 1)
    s_asp  = np.clip((CFG["TOMATO_MAX_ASPECT"] - metrics["aspect"]) / (CFG["TOMATO_MAX_ASPECT"] - 1.0), 0, 1)

    # Promedio ponderado
    return float(0.35 * s_circ + 0.35 * s_sol + 0.30 * s_asp)

def mask_from_contour(shape, cnt):
    m = np.zeros(shape[:2], dtype=np.uint8)
    cv2.drawContours(m, [cnt], -1, 255, -1)
    return m

## 4. Segmentación Inteligente

Aquí ocurre la magia de separar la fruta del fondo. En lugar de buscar un color específico, usamos una **estrategia de diferencia de fondo**.
1.  Asumimos que los bordes de la imagen son "fondo".
2.  Calculamos el promedio de color de esos bordes.
3.  Todo lo que sea muy diferente a ese color promedio, se considera "objeto de interés".

Esto permite que el sistema funcione tanto en una mesa blanca como en una mesa negra.

In [ ]:
def build_object_mask(img_bgr):
    blur = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    lab = cv2.cvtColor(blur, cv2.COLOR_BGR2LAB) # Usamos Lab para mejor percepción de color
    
    h, w = img_bgr.shape[:2]
    m = int(min(h, w) * CFG["BORDER_FRAC"])
    
    # Creamos máscara de bordes
    border_mask = np.zeros((h, w), dtype=np.uint8)
    border_mask[:m, :] = 1; border_mask[-m:, :] = 1
    border_mask[:, :m] = 1; border_mask[:, -m:] = 1
    
    # Color promedio del fondo
    A = lab[:, :, 1]; B = lab[:, :, 2]
    bgA = np.mean(A[border_mask == 1])
    bgB = np.mean(B[border_mask == 1])
    
    # Distancia de color
    dist = np.sqrt((A.astype(np.float32) - bgA)**2 + (B.astype(np.float32) - bgB)**2)
    dist_u8 = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # Binarización y limpieza
    _, mask = cv2.threshold(dist_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    k = CFG["KERNEL"]
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=CFG["CLOSE_IT"])
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=CFG["OPEN_IT"])

    return mask

## 5. Análisis Biológico: Madurez y Cáliz

Determinamos la madurez no por el promedio de color, sino contando píxeles.
* Si predomina el rojo -> **Maduro**.
* Si predomina el verde -> **Inmaduro**.

Además, buscamos el **Cáliz** solo en la parte superior del objeto detectado para confirmar que es un jitomate.

In [ ]:
def calyx_green_ratio(img_bgr, fruit_mask, cnt):
    x, y, w, h = cv2.boundingRect(cnt)
    top_h = int(h * CFG["CALYX_TOP_FRAC"]) # Solo miramos arriba
    if top_h < 5: return 0.0
    
    roi = img_bgr[y:y+top_h, x:x+w]
    roi_mask = fruit_mask[y:y+top_h, x:x+w]
    
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    lower = np.array([CFG["CALYX_H_MIN"], CFG["CALYX_MIN_S"], CFG["CALYX_MIN_V"]])
    upper = np.array([CFG["CALYX_H_MAX"], 255, 255])
    
    green_mask = cv2.inRange(hsv, lower, upper)
    valid_green = cv2.bitwise_and(green_mask, green_mask, mask=roi_mask)
    
    total = np.count_nonzero(roi_mask)
    if total == 0: return 0.0
    return np.count_nonzero(valid_green) / total

def classify_ripeness(img_bgr, fruit_mask):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    H = hsv[:,:,0]
    valid_pixels = H[fruit_mask > 0]
    total = len(valid_pixels)
    
    if total < 100: return "DESCONOCIDO", CFG["C_GRAY"]
    
    # Contamos píxeles por rango
    n_red = np.sum((valid_pixels <= CFG["H_RED1_MAX"]) | (valid_pixels >= CFG["H_RED2_MIN"]))
    n_green = np.sum((valid_pixels >= CFG["H_GREEN_MIN"]) & (valid_pixels <= CFG["H_GREEN_MAX"]))
    n_trans = np.sum((valid_pixels > CFG["H_ORANGE_MIN"]) & (valid_pixels <= CFG["H_YELLOW_MAX"]))
    
    p_red, p_green, p_trans = n_red/total, n_green/total, n_trans/total
    
    if p_red > p_green and p_red > p_trans: return "MADURO", CFG["C_RED"]
    elif p_green > p_red and p_green > p_trans: return "INMADURO", CFG["C_GREEN"]
    else: return "TRANSICION", CFG["C_ORANGE"]

def draw_results(img, cnt, label, color, score):
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.drawContours(img, [cnt], -1, color, 2)
    cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
    text = f"{label} ({score:.2f})"
    cv2.putText(img, text, (x, y-10), CFG["FONT"], 0.6, color, 2)

## 6. Ejecución Principal

Esta celda inicia la cámara web.
* Presiona **'q'** (con la ventana de video seleccionada) para detener el programa.
* **Nota:** Si usas Google Colab, la función `cv2.imshow` no funcionará directamente; necesitarás usar `from google.colab.patches import cv2_imshow`. Este código está diseñado para Jupyter local.

In [ ]:
def main():
    cap = cv2.VideoCapture(0) 
    print("Iniciando cámara... Presiona 'q' en la ventana de video para salir.")
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame = resize_to_width(frame, CFG["TARGET_VIEW_W"])
        output = frame.copy()
        mask = build_object_mask(frame)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        tomato_count = 0
        
        for c in cnts:
            if cv2.contourArea(c) < CFG["MIN_AREA"]: continue
            
            metrics = contour_metrics(c)
            shape_score = tomato_shape_score(metrics)
            fruit_mask = mask_from_contour(frame.shape, c)
            calyx_ratio = calyx_green_ratio(frame, fruit_mask, c)
            
            # Ponderación: Si tiene cáliz, es casi seguro un jitomate
            if CFG["USE_CALYX_CHECK"]:
                total_score = (0.7 * shape_score) + (0.3 * (calyx_ratio * 10))
            else:
                total_score = shape_score

            is_tomato = total_score > 0.55
            
            if is_tomato:
                label, color = classify_ripeness(frame, fruit_mask)
                draw_results(output, c, label, color, total_score)
                tomato_count += 1
            else:
                draw_results(output, c, "No Tomato", CFG["C_GRAY"], total_score)
        
        cv2.putText(output, f"Jitomates: {tomato_count}", (10, 30), CFG["FONT"], 1, (0, 255, 255), 2)
        cv2.imshow("Analisis de Fruta", output)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()